# Bronze Positions

Loads data from `bundesliga-2022-2023.batch.raw_positions` (timemajor tracking data) and shows sample rows.

## Bronze Transformation Steps

Input: `raw_positions` (15 columns, \~23.7M rows) → Output: `bronze_positions` (36 columns, \~23.7M rows)

| Step | Cell | Action | Columns Added / Modified |
| --- | --- | --- | --- |
| 0 | 3 | Load `raw_positions` from Delta table | 15 input columns loaded |
| 1 | 4 | Compute attacking direction per team per half via goalkeeper X position | `directions` DataFrame (temp, 28 rows) |
| 2 | 5 | Build `create_map` lookup from directions (28 entries, key = `match_id\|game_section\|team_id`) | `attack_map` (temp map expression) |
| 3 | 6 | Add `attacking_direction` to every row. BALL rows resolved via `ball_possession` (1=home, 2=guest) → possessing team's direction | `attacking_direction` added (col 16) |
| 4 | 7 | Add `x_norm`, `y_norm` — attack-normalized coordinates (0 = own goal, pitch dim = opponent goal). Flips based on `attacking_direction` | `x_norm`, `y_norm` added (cols 17–18) |
| 5 | 8 | Add `ball_distance` — Euclidean distance from each entity to the ball at the same frame (window over `match_id`, `frame_id`) | `ball_distance` added (col 19) |
| 6 | 9 | Add `pitch_zone` (9-cell grid: 3 X-thirds × 3 Y-lanes) and `zone_id` (1–9) based on `x_norm` / `y_norm` | `pitch_zone`, `zone_id` added (cols 20–21) |
| 7 | 10 | Add `prev_` columns via `lag()` window (partitioned by `match_id`, `person_id`, ordered by `frame_id`). Nullified on non-consecutive frames. First-frame rows filtered out (394 rows). | `prev_frame_id`, `prev_timestamp`, `prev_x`, `prev_y`, `prev_z`, `prev_speed`, `prev_distance`, `prev_acceleration`, `prev_ball_possession`, `prev_ball_status`, `prev_attacking_direction`, `prev_x_norm`, `prev_y_norm`, `prev_ball_distance`, `prev_pitch_zone` (cols 22–36) |

**Output schema (36 columns):**

| # | Column | Type | Source |
| --- | --- | --- | --- |
| 1–15 | `frame_id` ... `ball_status` | various | Passthrough from `raw_positions` |
| 16 | `attacking_direction` | int | Step 3 — create_map lookup + BALL resolution |
| 17–18 | `x_norm`, `y_norm` | double | Step 4 — attack-normalized coordinates |
| 19 | `ball_distance` | double | Step 5 — Euclidean distance to ball |
| 20–21 | `pitch_zone`, `zone_id` | string, int | Step 6 — 9-cell pitch classification |
| 22–36 | `prev_*` (15 cols) | various | Step 7 — lag() window, nullified on gaps |

In [0]:
# ── Load raw_positions and show sample rows ──

from pyspark.sql.functions import col

bronze_df = spark.table("`bundesliga-2022-2023`.batch.raw_positions").drop("frame_number")

print(f"Table: `bundesliga-2022-2023`.batch.raw_positions")
print(f"  Columns: {len(bronze_df.columns)}")
print(f"  Rows: {bronze_df.count():,}")
print(f"  Schema:")
for f in bronze_df.schema.fields:
    print(f"    {f.name}: {f.dataType}")

print("\n=== Sample: ball rows (first 10) ===")
display(
    bronze_df.filter(col("team_id") == "BALL")
    .select("match_id", "frame_id", "timestamp", "x", "y", "z", "speed", "distance", "acceleration", "ball_possession", "ball_status")
    .orderBy("match_id", "frame_id")
    .limit(10)
)

print("\n=== Sample: player rows (first 10) ===")
display(
    bronze_df.filter(col("team_id") != "BALL")
    .filter(col("team_id") != "referee")
    .select("match_id", "frame_id", "timestamp", "team_id", "person_id", "x", "y", "speed", "distance", "acceleration")
    .orderBy("match_id", "frame_id")
    .limit(10)
)

print("\n=== Sample: all entities at one frame ===")
sample_frame = bronze_df.filter(
    (col("match_id") == "DFL-MAT-J03WMX") &
    (col("frame_id") == 10000)
).select("team_id", "person_id", "x", "y", "z", "speed", "distance", "acceleration", "ball_possession", "ball_status")
display(sample_frame.orderBy(col("team_id"), col("person_id")))
print(f"\nEntities at this frame: {sample_frame.count()}")

In [0]:
# ── Attacking direction function ──
# Determines attacking direction for each team in each half.
#   +1 = attacks left-to-right (toward positive X)
#   -1 = attacks right-to-left (toward negative X)
#
# Logic: the goalkeeper is the player closest to their own goal.
# If GK avg_x < 0 → team's own goal is at negative X → they attack toward positive X → direction = +1
# If GK avg_x > 0 → team's own goal is at positive X → they attack toward negative X → direction = -1
#
# Coordinates in raw_positions are ABSOLUTE (fixed to pitch). Teams switch sides at halftime.

from pyspark.sql.functions import col, avg, when, broadcast, row_number, abs as spark_abs
from pyspark.sql.window import Window


def get_attack_directions(positions_df, match_info_df):
    """
    Returns a DataFrame with columns:
      match_id, game_section, team_id, attack_direction, gk_avg_x
    where attack_direction is +1 (left-to-right) or -1 (right-to-left).

    Parameters:
      positions_df  – the raw_positions Delta table
      match_info_df – the match_info Delta table (for home/guest team IDs)
    """
    # Get all valid team IDs per match
    teams = match_info_df.select(
        col("match_id"),
        col("home_team_id").alias("team_id"),
    ).unionAll(
        match_info_df.select(
            col("match_id"),
            col("guest_team_id").alias("team_id"),
        )
    ).distinct()

    # For each match+half+team, compute each player's avg X in the first 200 frames
    # First half starts at frame 10000, second half at frame 100000
    half_starts = [("firstHalf", 10000), ("secondHalf", 100000)]

    gk_rows = []
    for half, start_frame in half_starts:
        player_avgs = positions_df.filter(
            col("game_section") == half
        ).filter(
            col("frame_id").between(start_frame, start_frame + 200)
        ).join(
            broadcast(teams), ["match_id", "team_id"]
        ).groupBy(
            "match_id", "game_section", "team_id", "person_id"
        ).agg(
            avg("x").alias("avg_x")
        )

        # GK = player with the largest |avg_x| per team (most extreme position = closest to goal)
        w = Window.partitionBy("match_id", "game_section", "team_id").orderBy(spark_abs(col("avg_x")).desc())
        gk = player_avgs.withColumn("rn", row_number().over(w)).filter(col("rn") == 1).drop("rn")
        gk_rows.append(gk)

    all_gk = gk_rows[0].unionByName(gk_rows[1])

    # attack_direction: if GK at negative X → attacks positive → +1; if GK at positive X → attacks negative → -1
    directions = all_gk.select(
        "match_id",
        "game_section",
        "team_id",
        when(col("avg_x") < 0, 1).otherwise(-1).alias("attack_direction"),
        col("avg_x").alias("gk_avg_x"),
    )

    return directions


def add_attack_direction(positions_df, directions_df):
    """
    Joins attack_direction onto positions_df.
    Adds columns: attack_direction, x_attacking (x * attack_direction)
    """
    return positions_df.join(
        directions_df.select("match_id", "game_section", "team_id", "attack_direction"),
        on=["match_id", "game_section", "team_id"],
        how="left"
    ).withColumn(
        "x_attacking", col("x") * col("attack_direction")
    )


# ── Run the function and show results ──

positions = spark.table("`bundesliga-2022-2023`.batch.raw_positions")
match_info = spark.table("`bundesliga-2022-2023`.batch.match_info")

directions = get_attack_directions(positions, match_info)

print("=== Attacking direction per team per half (all 7 matches) ===")
directions.orderBy("match_id", "game_section", "team_id").show(28)

In [0]:
# ── Attack directions as create_map lookup ──
# Instead of a join, build a single create_map expression that maps
# "match_id|game_section|team_id" → attack_direction (28 entries, 7 matches × 2 halves × 2 teams)
# This is efficient for small lookup tables and avoids join overhead.

from pyspark.sql.functions import create_map, lit, concat_ws, col

# Collect directions to a Python dict (only 28 rows)
dir_rows = directions.collect()

# Build create_map expression: map("key1", val1, "key2", val2, ...)
map_args = []
for r in dir_rows:
    key = f"{r['match_id']}|{r['game_section']}|{r['team_id']}"
    map_args.append(lit(key))
    map_args.append(lit(r['attack_direction']))

attack_map = create_map(*map_args)

print(f"Map entries: {len(dir_rows)}")
print("=== Sample map keys ===")
for r in dir_rows[:6]:
    key = f"{r['match_id']}|{r['game_section']}|{r['team_id']}"
    print(f"  {key} → {r['attack_direction']}")

# ── Usage: add attack_direction to positions without a join ──
positions_with_dir = positions.withColumn(
    "_lookup_key", concat_ws("|", "match_id", "game_section", "team_id")
).withColumn(
    "attack_direction", attack_map[col("_lookup_key")]
).withColumn(
    "x_attacking", col("x") * col("attack_direction")
).drop("_lookup_key")

print("\n=== Sample: positions with attack_direction (match DFL-MAT-J03WMX, frame 10000) ===")
display(
    positions_with_dir.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10000)
    ).select("team_id", "person_id", "x", "attack_direction", "x_attacking")
    .orderBy(col("team_id"), col("person_id"))
)

In [0]:
# ── Add attack_direction using element_at on create_map ──
# For BALL rows: resolve ball_possession (1=home, 2=guest) to the actual team_id,
# then look up that team's attacking_direction. This ensures the ball's x_norm/y_norm
# and pitch_zone are computed in the same coordinate system as the team in possession.

from pyspark.sql.functions import col, concat_ws, element_at, when, create_map, lit

# attack_map already built in previous cell from directions

# Build home/guest team_id maps: match_id → home_team_id / guest_team_id
mi_rows = match_info.select("match_id", "home_team_id", "guest_team_id").collect()
home_args, guest_args = [], []
for r in mi_rows:
    home_args += [lit(r["match_id"]), lit(r["home_team_id"])]
    guest_args += [lit(r["match_id"]), lit(r["guest_team_id"])]
home_map = create_map(*home_args)
guest_map = create_map(*guest_args)

# For BALL rows: resolve ball_possession to the team_id of the team in possession
# For non-BALL rows: use their own team_id
lookup_team_id = when(
    col("team_id") == "BALL",
    when(col("ball_possession") == 1, element_at(home_map, col("match_id")))
    .when(col("ball_possession") == 2, element_at(guest_map, col("match_id")))
    .otherwise(lit(None))
).otherwise(col("team_id"))

bronze_df = positions.drop("frame_number").withColumn(
    "attacking_direction",
    element_at(attack_map, concat_ws("|", col("match_id"), col("game_section"), lookup_team_id))
)

print("=== Sample: bronze_df with attacking_direction (match DFL-MAT-J03WMX, frame 10000) ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10000)
    ).select("team_id", "person_id", "x", "ball_possession", "attacking_direction")
    .orderBy(col("team_id"), col("person_id"))
)

print("\n=== Verify: both halves (match DFL-MAT-J03WMX, first frame of each half) ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id").isin([10000, 100000])) &
        (col("team_id") != "referee")
    ).select("game_section", "team_id", "person_id", "x", "attacking_direction")
    .orderBy("game_section", "team_id", "person_id")
)

In [0]:
# ── Add x_norm: attack-normalized X coordinate ──
# x_norm always goes from 0 (own goal) to pitch_length (opponent goal).
#
# If attacking_direction = +1 (left-to-right):  x_norm = x + pitch_x/2
# If attacking_direction = -1 (right-to-left): x_norm = pitch_x - (x + pitch_x/2)
#
# pitch_x comes from match_info via create_map + element_at (no join).

from pyspark.sql.functions import col, when, element_at, create_map, lit, round

# Build pitch_x and pitch_y maps from match_info: match_id → pitch_x / pitch_y (7 entries each)
pitch_rows = match_info.select("match_id", "pitch_x", "pitch_y").collect()
pitch_x_args, pitch_y_args = [], []
for r in pitch_rows:
    pitch_x_args += [lit(r["match_id"]), lit(r["pitch_x"])]
    pitch_y_args += [lit(r["match_id"]), lit(r["pitch_y"])]

pitch_x_map = create_map(*pitch_x_args)
pitch_y_map = create_map(*pitch_y_args)

bronze_df = bronze_df.withColumn(
    "x_norm",
    round(
        when(
            col("attacking_direction") == 1,
            col("x") + element_at(pitch_x_map, col("match_id")) / 2
        ).otherwise(
            element_at(pitch_x_map, col("match_id")) - (col("x") + element_at(pitch_x_map, col("match_id")) / 2)
        ),
        2
    )
).withColumn(
    "y_norm",
    round(
        when(
            col("attacking_direction") == 1,
            col("y") + element_at(pitch_y_map, col("match_id")) / 2
        ).otherwise(
            element_at(pitch_y_map, col("match_id")) - (col("y") + element_at(pitch_y_map, col("match_id")) / 2)
        ),
        2
    )
)

print("=== Sample: bronze_df with x_norm (match DFL-MAT-J03WMX, frame 10000) ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10000)
    ).select("team_id", "person_id", "x", "y", "attacking_direction", "x_norm", "y_norm")
    .orderBy(col("team_id"), col("person_id"))
)

print("\n=== Verify: both halves — GK should always have x_norm near 0 ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id").isin([10000, 100000])) &
        (col("person_id").isin(["DFL-OBJ-0002HE", "DFL-OBJ-0002DR"]))
    ).select("game_section", "team_id", "person_id", "x", "y", "attacking_direction", "x_norm", "y_norm")
    .orderBy("game_section", "team_id")
)

In [0]:
# ── Add ball_distance: Euclidean distance from each entity to the ball at the same frame ──
# Uses a window function to get the ball's x/y for each (match_id, frame_number)
# then computes sqrt((x - ball_x)^2 + (y - ball_y)^2) for every row.
# Raw x/y are used (not x_norm/y_norm) since distance is coordinate-system invariant.

from pyspark.sql.functions import col, when, max as spark_max, sqrt, pow as spark_pow, round
from pyspark.sql.window import Window

w = Window.partitionBy("match_id", "frame_id")

bronze_df = bronze_df.withColumn(
    "ball_x", spark_max(when(col("team_id") == "BALL", col("x"))).over(w)
).withColumn(
    "ball_y", spark_max(when(col("team_id") == "BALL", col("y"))).over(w)
).withColumn(
    "ball_distance",
    round(sqrt(spark_pow(col("x") - col("ball_x"), 2) + spark_pow(col("y") - col("ball_y"), 2)), 2)
).drop("ball_x", "ball_y")

print("=== Sample: bronze_df with ball_distance (match DFL-MAT-J03WMX, frame 10000) ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("frame_id") == 10000)
    ).select("team_id", "person_id", "x", "y", "ball_distance")
    .orderBy(col("ball_distance").asc())
)

In [0]:
# ── Add pitch_zone: 9-cell classification based on x_norm / y_norm ──
# Pitch divided into 3 thirds (X: first-third 0-35, second-third 35-70, final-third 70-105)
# and 3 lanes (Y: left 0-22.67, centre 22.67-45.33, right 45.33-68)
# Applied to x_norm / y_norm (attack-normalized, 0 to pitch dimensions).
# Every entity (players + ball) gets its own zone classification at row level.

from pyspark.sql.functions import col, when, lit, concat, create_map

PITCH_X = 105.0
PITCH_Y = 68.0

x_third = (
    when(col("x_norm") < PITCH_X / 3, lit("first-third"))
    .when(col("x_norm") < 2 * PITCH_X / 3, lit("second-third"))
    .otherwise(lit("final-third"))
)

y_zone = (
    when(col("y_norm") < PITCH_Y / 3, lit("left"))
    .when(col("y_norm") < 2 * PITCH_Y / 3, lit("centre"))
    .otherwise(lit("right"))
)

zone_map = create_map(
    lit("first-third_left"),    lit(1),
    lit("first-third_centre"),  lit(2),
    lit("first-third_right"),   lit(3),
    lit("second-third_left"),   lit(4),
    lit("second-third_centre"), lit(5),
    lit("second-third_right"),  lit(6),
    lit("final-third_left"),    lit(7),
    lit("final-third_centre"),  lit(8),
    lit("final-third_right"),   lit(9),
)

bronze_df = bronze_df.withColumn(
    "pitch_zone", concat(x_third, lit("_"), y_zone)
).withColumn(
    "zone_id", zone_map[col("pitch_zone")]
)

print("=== Sample: bronze_df with pitch_zone (match DFL-MAT-J03WMX, frame 10001) ===")
bronze_df.filter((col("match_id") == "DFL-MAT-J03WMX") & (col("frame_id") == 10001)) \
    .select("team_id", "person_id", "x_norm", "y_norm", "pitch_zone", "zone_id") \
    .show(30, truncate=False)

print("=== Zone distribution (all entities) ===")
bronze_df.groupBy("pitch_zone").count().orderBy("pitch_zone").show(truncate=False)

In [0]:
# ── Add prev_ columns: previous frame values for each (match_id, person_id) ──
# Uses lag() over a window partitioned by match_id + person_id, ordered by frame_number.
# Adds prev_ prefix for: frame_number, timestamp, x, y, z, speed, distance, acceleration,
# ball_possession, ball_status, attacking_direction, x_norm, y_norm, ball_distance

from pyspark.sql.functions import col, lag, when
from pyspark.sql.window import Window

w = Window.partitionBy("match_id", "person_id").orderBy("frame_id")

prev_cols = [
    "frame_id", "timestamp", "x", "y", "z",
    "speed", "distance", "acceleration",
    "ball_possession", "ball_status",
    "attacking_direction", "x_norm", "y_norm", "ball_distance",
    "pitch_zone"
]

for c in prev_cols:
    bronze_df = bronze_df.withColumn(f"prev_{c}", lag(col(c)).over(w))

# Capture non-consecutive flag BEFORE nullifying prev_frame_id
# (prev_frame_id is modified in the loop below, which would break the condition)
bronze_df = bronze_df.withColumn(
    "_non_consecutive",
    col("prev_frame_id").isNull() | (col("prev_frame_id") != (col("frame_id") - 1))
)

# Nullify prev_ columns when frames are not consecutive (prev_frame_id != frame_id - 1)
# This prevents cross-half leaks at halftime boundaries
for c in prev_cols:
    bronze_df = bronze_df.withColumn(
        f"prev_{c}", when(col("_non_consecutive"), None).otherwise(col(f"prev_{c}"))
    )

bronze_df = bronze_df.drop("_non_consecutive")

# Filter out rows where prev_frame_id is NULL (first frame of each half).
# These rows have no previous-frame data and are already referenced
# by the next frame (e.g. frame 10000 is prev_ of 10001).
bronze_df = bronze_df.filter(col("prev_frame_id").isNotNull())

print("=== Sample: bronze_df with prev_ columns (match DFL-MAT-J03WMX, person DFL-OBJ-0027G6, first 5 frames) ===")
display(
    bronze_df.filter(
        (col("match_id") == "DFL-MAT-J03WMX") &
        (col("person_id") == "DFL-OBJ-0027G6")
    ).select(
        "frame_id", "x", "prev_x",
        "y", "prev_y",
        "speed", "prev_speed",
        "ball_distance", "prev_ball_distance"
    ).orderBy("frame_id")
    .limit(5)
)

print(f"\nTotal columns in bronze_df: {len(bronze_df.columns)}")

In [0]:
# ── Bronze_df summary ──

from pyspark.sql.functions import col, countDistinct

n_rows = bronze_df.count()
n_cols = len(bronze_df.columns)
n_matches = bronze_df.select("match_id").distinct().count()
n_persons = bronze_df.select("person_id").distinct().count()

print(f"=== bronze_df summary ===")
print(f"  Rows:      {n_rows:,}")
print(f"  Columns:   {n_cols}")
print(f"  Matches:   {n_matches}")
print(f"  Entities:  {n_persons}")

print(f"\n=== Columns ({n_cols}) ===")
print(f"{'#':<4} {'Column':<30} {'Type':<15}")
print("-" * 51)
for i, f in enumerate(bronze_df.schema.fields, 1):
    print(f"{i:<4} {f.name:<30} {str(f.dataType):<15}")

In [0]:
# ── Verify frame boundaries: no cross-half or cross-match leakage in prev_ columns ──

from pyspark.sql.functions import col, count as spark_count, when, max as spark_max

# 1. Check that frame_id 10000 exists in firstHalf for all matches
first_half_starts = bronze_df.filter(
    (col("frame_id") == 10000) & (col("game_section") == "firstHalf")
).select("match_id").distinct()

# 2. Check that frame_id 100000 exists in secondHalf for all matches
second_half_starts = bronze_df.filter(
    (col("frame_id") == 100000) & (col("game_section") == "secondHalf")
).select("match_id").distinct()

n_matches = bronze_df.select("match_id").distinct().count()
n_fh = first_half_starts.count()
n_sh = second_half_starts.count()

print(f"Total matches: {n_matches}")
print(f"Matches with frame_id 10000 in firstHalf:  {n_fh} / {n_matches}")
print(f"Matches with frame_id 100000 in secondHalf: {n_sh} / {n_matches}")

# 3. Check prev_frame_id at frame 10000 (firstHalf start) — should be NULL for all rows
fh_prev_nulls = bronze_df.filter(
    (col("frame_id") == 10000) & (col("game_section") == "firstHalf")
).agg(
    spark_count(when(col("prev_frame_id").isNull(), 1)).alias("null_count"),
    spark_count(col("prev_frame_id")).alias("non_null_count")
).collect()[0]
print(f"\n=== frame 10000 (firstHalf start): prev_frame_id ===")
print(f"  NULL prev_frame_id:    {fh_prev_nulls['null_count']}")
print(f"  Non-NULL prev_frame_id: {fh_prev_nulls['non_null_count']}")

# 4. Check prev_frame_id at frame 100000 (secondHalf start) — should it be NULL?
sh_prev_vals = bronze_df.filter(
    (col("frame_id") == 100000) & (col("game_section") == "secondHalf")
).select("prev_frame_id").distinct()
print(f"\n=== frame 100000 (secondHalf start): distinct prev_frame_id values ===")
sh_prev_vals.show()

# 5. Check if prev_frame_id at frame 100000 points to last frame of firstHalf (cross-half leak)
# Find the max frame_id per match per half
max_frames = bronze_df.groupBy("match_id", "game_section").agg(
    spark_max("frame_id").alias("max_frame")
).orderBy("match_id", "game_section")
print(f"\n=== Max frame_id per match per half ===")
max_frames.show(14)

# 6. Check: does prev_frame_id at 100000 equal the max frame_id of firstHalf for any match?
fh_max = max_frames.filter(col("game_section") == "firstHalf").select(
    col("match_id"), col("max_frame").alias("fh_last_frame")
)
sh_prev = bronze_df.filter(
    (col("frame_id") == 100000) & (col("game_section") == "secondHalf")
).select("match_id", col("prev_frame_id").alias("sh_prev_frame")).distinct()

cross_half = fh_max.join(sh_prev, "match_id").filter(
    col("fh_last_frame") == col("sh_prev_frame")
)
n_cross = cross_half.count()
print(f"\n=== Cross-half leak check ===")
print(f"  Matches where prev_frame_id at 100000 == last frame of firstHalf: {n_cross}")
if n_cross > 0:
    print("  \u26a0\ufe0f  CROSS-HALF LEAK DETECTED:")
    cross_half.orderBy("match_id").show()
else:
    print("  \u2705 No cross-half leak — prev_ at secondHalf start is NULL or points elsewhere")

# 7. Verify no cross-match leak: prev_frame_id should never reference a frame from a different match
# (impossible by window partition, but verify for safety)
print(f"\n=== Cross-match leak check ===")
print(f"  Window is partitioned by (match_id, person_id) \u2192 cross-match leak is impossible by design \u2705")

In [0]:
# ── Verify row counts and frame completeness ──
# 1. raw_positions vs bronze_df row count (should be equal — all transforms are additive)
# 2. Check for missing frames (gaps in frame_id sequence) per match per half
# 3. Check entity count per frame is consistent (no frame missing entities)

from pyspark.sql.functions import col, count as spark_count, countDistinct, min as spark_min, max as spark_max, lag, when, sum as spark_sum
from pyspark.sql.window import Window

# ── 1. Row count comparison ──
raw = spark.table("`bundesliga-2022-2023`.batch.raw_positions")
raw_count = raw.count()
bronze_count = bronze_df.count()

print("=== 1. Row count comparison ===")
print(f"  raw_positions:  {raw_count:,}")
print(f"  bronze_df:      {bronze_count:,}")
print(f"  Match:          {'\u2705 YES' if raw_count == bronze_count else '\u274c NO — rows lost!'}")

# ── 2. Check for missing frames (gaps in frame_id sequence) ──
# Get distinct frame_ids per match per half
frames_per_half = bronze_df.groupBy("match_id", "game_section").agg(
    spark_count("frame_id").alias("n_frame_rows"),
    countDistinct("frame_id").alias("n_distinct_frames"),
    spark_min("frame_id").alias("min_frame"),
    spark_max("frame_id").alias("max_frame")
).orderBy("match_id", "game_section")

# For each match+half, expected frame count = max_frame - min_frame + 1
# But frame_id may restart at 100000 for secondHalf, so we check consecutive within each half
frame_stats = frames_per_half.withColumn(
    "expected_frames", col("max_frame") - col("min_frame") + 1
).withColumn(
    "missing_frames", col("expected_frames") - col("n_distinct_frames"))

print(f"\n=== 2. Frame completeness per match per half ===")
print(f"{'match_id':<20} {'half':<12} {'frames':>8} {'distinct':>8} {'min':>8} {'max':>8} {'expected':>8} {'missing':>8}")
print("-" * 90)
for r in frame_stats.collect():
    missing_str = str(r["missing_frames"]) if r["missing_frames"] != 0 else "0 \u2705"
    print(f"{r['match_id']:<20} {r['game_section']:<12} {r['n_frame_rows']:>8,} {r['n_distinct_frames']:>8,} {r['min_frame']:>8,} {r['max_frame']:>8,} {r['expected_frames']:>8,} {missing_str:>8}")

# ── 3. Check for duplicate frame_ids within a match+half (same frame appearing multiple times) ──
dup_frames = frame_stats.filter(col("n_frame_rows") != col("n_distinct_frames") * (col("n_frame_rows") / col("n_distinct_frames")))
print(f"\n=== 3. Duplicate frame check ===")
# Check if entity count per frame is consistent
entities_per_frame = bronze_df.groupBy("match_id", "game_section", "frame_id").agg(
    spark_count("person_id").alias("n_entities")
)
ent_stats = entities_per_frame.groupBy("match_id", "game_section").agg(
    spark_min("n_entities").alias("min_entities"),
    spark_max("n_entities").alias("max_entities"),
    countDistinct("n_entities").alias("distinct_entity_counts")
).orderBy("match_id", "game_section")

print(f"{'match_id':<20} {'half':<12} {'min_ent':>8} {'max_ent':>8} {'distinct_counts':>15}")
print("-" * 70)
for r in ent_stats.collect():
    flag = " \u2705" if r["distinct_entity_counts"] == 1 else " \u26a0\ufe0f variable"
    print(f"{r['match_id']:<20} {r['game_section']:<12} {r['min_entities']:>8} {r['max_entities']:>8} {r['distinct_entity_counts']:>15}{flag}")

# ── 4. Verify consecutive frame pairs exist (for prev_ columns) ──
# For each match+half, get sorted distinct frame_ids and check if each is +1 from previous
w_frames = Window.partitionBy("match_id", "game_section").orderBy("frame_id")
frame_seq = bronze_df.select("match_id", "game_section", "frame_id").distinct().withColumn(
    "prev_frame", lag("frame_id").over(w_frames)
).withColumn(
    "gap", when(col("frame_id") - col("prev_frame") != 1, 1).otherwise(0))

gaps = frame_seq.filter(col("gap") == 1).select("match_id", "game_section", "frame_id", "prev_frame")
n_gaps = gaps.count()
print(f"\n=== 4. Consecutive frame pairs check ===")
print(f"  Total gaps found: {n_gaps}")
if n_gaps > 0:
    print(f"  \u26a0\ufe0f  Gaps detected (frame_id jumps > 1):")
    gaps.orderBy("match_id", "game_section", "frame_id").show(50, truncate=False)
else:
    print(f"  \u2705 All frames are consecutive — no missing frame_ids within any half")

In [0]:
# ── Null value audit across all columns ──

from pyspark.sql.functions import col, count as spark_count, when, sum as spark_sum

total_rows = bronze_df.count()

null_counts = bronze_df.select([
    spark_sum(when(col(c).isNull(), 1).otherwise(0)).alias(c)
    for c in bronze_df.columns
]).collect()[0]

print(f"Total rows: {total_rows:,}")
print(f"\n{'Column':<30} {'Nulls':>12} {'Null %':>8}  Status")
print("-" * 65)
for c in bronze_df.columns:
    n_null = null_counts[c]
    pct = (n_null / total_rows) * 100
    if n_null == 0:
        status = "\u2705 clean"
    elif n_null == total_rows:
        status = "\u274c all null"
    else:
        status = "\u26a0\ufe0f has nulls"
    print(f"{c:<30} {n_null:>12,} {pct:>7.2f}%  {status}")

In [0]:
# ── Breakdown of 394 filtered rows (prev_frame_id = NULL) ──
# The 394 rows filtered out in cell 8 come from two sources:
#   1. First frame of each half (14 frames × entities per frame = 333)
#   2. Late-entrant entities whose first appearance is mid-match (61 substitutes)

from pyspark.sql.functions import col, count as spark_count, min as spark_min

raw = spark.table("`bundesliga-2022-2023`.batch.raw_positions")

# ── 1. First-frame entity counts ──
first_frames = raw.filter(
    (col("frame_id") == 10000) | (col("frame_id") == 100000)
).groupBy("match_id", "game_section", "frame_id").agg(
    spark_count("*").alias("n_entities")
).orderBy("match_id", "game_section")

print("=== 1. First-frame entity counts (frame 10000 / 100000) ===")
print(f"{'match_id':<20} {'half':<12} {'frame_id':>10} {'entities':>8}")
print("-" * 55)
total_first = 0
for r in first_frames.collect():
    print(f"{r['match_id']:<20} {r['game_section']:<12} {r['frame_id']:>10,} {r['n_entities']:>8}")
    total_first += r['n_entities']
print(f"{'':>20} {'':<12} {'TOTAL':>10} {total_first:>8}")

# ── 2. Late-entrant entities (first appearance NOT at first frame) ──
first_appearance = raw.groupBy("match_id", "person_id").agg(
    spark_min("frame_id").alias("first_frame")
).filter(
    (col("first_frame") != 10000) & (col("first_frame") != 100000)
)

late_entrants = first_appearance.orderBy("match_id", "first_frame")
late_count = late_entrants.count()

print(f"\n=== 2. Late-entrant entities (first appearance mid-match) ===")
print(f"{'match_id':<20} {'person_id':<20} {'first_frame':>10}")
print("-" * 55)
for r in late_entrants.collect():
    print(f"{r['match_id']:<20} {r['person_id']:<20} {r['first_frame']:>10,}")
print(f"{'':>20} {'TOTAL':>20} {late_count:>10}")

# ── Summary ──
print(f"\n=== Summary ===")
print(f"  First-frame rows:    {total_first}")
print(f"  Late-entrant rows:   {late_count}")
print(f"  Total filtered:      {total_first + late_count}")
print(f"  These rows are already referenced as prev_ in the next frame.")

In [0]:
# ── Save bronze_df as Delta table ──
# Target: `bundesliga-2022-2023`.batch.bronze_positions
# Partitioned by match_id for efficient per-match queries.

# Drop existing table first to avoid UC metadata conflicts with schema changes
spark.sql("DROP TABLE IF EXISTS `bundesliga-2022-2023`.batch.bronze_positions")

bronze_df.write \
    .mode("overwrite") \
    .option("overwriteSchema", "true") \
    .partitionBy("match_id") \
    .format("delta") \
    .saveAsTable("`bundesliga-2022-2023`.batch.bronze_positions")

print(f"Saved to `bundesliga-2022-2023`.batch.bronze_positions")
print(f"  Rows: {bronze_df.count():,}")
print(f"  Columns: {len(bronze_df.columns)}")

# Verify table exists and row count matches
saved = spark.table("`bundesliga-2022-2023`.batch.bronze_positions")
saved_count = saved.count()
print(f"  Verified rows: {saved_count:,}")
print(f"  Match: {'\u2705' if saved_count == bronze_df.count() else '\u274c'}")